# Practical 8: Implement a Game-Playing Agent Using the Minimax Algorithm

## Aim
Implement a Minimax-based intelligent agent that selects legal moves in Tic-Tac-Toe.

In [1]:
# Cell 1: Tic-Tac-Toe Game Representation and Minimax Algorithm Implementation

def format_grid(board):
    """Renders 3x3 board."""
    lines = []
    lines.append("   0   1   2")
    for r in range(3):
        row_str = [board[r*3 + c] for c in range(3)]
        lines.append(f"{r}  {' | '.join(row_str)}")
        if r < 2:
            lines.append("  ---+---+---")
    return "\n".join(lines)

def detect_winner(board):
    """Detects 'X' win, 'O' win, 'Draw', or ongoing game (None)."""
    win_lines = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8), # Rows
        (0, 3, 6), (1, 4, 7), (2, 5, 8), # Columns
        (0, 4, 8), (2, 4, 6)             # Diagonals
    ]
    for a, b, c in win_lines:
        if board[a] != ' ' and board[a] == board[b] == board[c]:
            return board[a]
    if ' ' not in board:
        return 'Draw'
    return None

def get_open_moves(board):
    """Returns indices of unoccupied cells."""
    return [i for i in range(9) if board[i] == ' ']

def minimax_score(board, depth, is_maximizing, ai_mark='O', human_mark='X'):
    """Minimax recursive evaluation with depth penalty."""
    outcome = detect_winner(board)
    if outcome == ai_mark:
        return 10 - depth
    elif outcome == human_mark:
        return depth - 10
    elif outcome == 'Draw':
        return 0

    if is_maximizing:
        best_val = float('-inf')
        for move in get_open_moves(board):
            board[move] = ai_mark
            score = minimax_score(board, depth + 1, False, ai_mark, human_mark)
            board[move] = ' '
            best_val = max(best_val, score)
        return best_val
    else:
        best_val = float('inf')
        for move in get_open_moves(board):
            board[move] = human_mark
            score = minimax_score(board, depth + 1, True, ai_mark, human_mark)
            board[move] = ' '
            best_val = min(best_val, score)
        return best_val

def select_best_move(board, ai_mark='O', human_mark='X'):
    """Picks best legal move for AI using minimax."""
    best_score = float('-inf')
    best_move = None
    for move in get_open_moves(board):
        board[move] = ai_mark
        score = minimax_score(board, 0, False, ai_mark, human_mark)
        board[move] = ' '
        if score > best_score:
            best_score = score
            best_move = move
    return best_move


In [2]:
# Cell 2: Scenario Demonstrations, Complete Automated Game, and Verification

print("=" * 65)
print("SCENARIO 1: IMMEDIATE WINNING MOVE SELECTION")
print("=" * 65)
# AI ('O') has positions 0 and 1; position 2 gives immediate victory
board_win = ['O', 'O', ' ', 'X', 'X', ' ', ' ', ' ', ' ']
print("Current Board:")
print(format_grid(board_win))
win_move = select_best_move(board_win, ai_mark='O', human_mark='X')
print(f"Minimax Selected Move: Cell {win_move} (Expected: 2 to win)")
board_win[win_move] = 'O'
print(format_grid(board_win))
assert win_move == 2, "Verification Failed: AI did not pick immediate winning move!"
assert detect_winner(board_win) == 'O', "Verification Failed: AI did not win!"
print("Verification: Immediate winning move selected and validated successfully.\n")

print("=" * 65)
print("SCENARIO 2: IMMEDIATE BLOCKING MOVE SELECTION")
print("=" * 65)
# Opponent ('X') has positions 3 and 4; position 5 threatens victory
board_threat = ['O', ' ', ' ', 'X', 'X', ' ', ' ', ' ', ' ']
print("Current Board:")
print(format_grid(board_threat))
block_move = select_best_move(board_threat, ai_mark='O', human_mark='X')
print(f"Minimax Selected Move: Cell {block_move} (Expected: 5 to block opponent)")
board_threat[block_move] = 'O'
print(format_grid(board_threat))
assert block_move == 5, "Verification Failed: AI did not block opponent's winning line!"
print("Verification: Threat blocked successfully.\n")

print("=" * 65)
print("SCENARIO 3: FULL SIMULATED GAMEPLAY (AI 'O' VS DETERMINISTIC 'X')")
print("=" * 65)
game_board = [' '] * 9
print("Initial Board:")
print(format_grid(game_board))
print("-" * 65)

current_turn = 'X'
turn_count = 1

while True:
    legal_moves = get_open_moves(game_board)
    if not legal_moves or detect_winner(game_board) is not None:
        break

    if current_turn == 'O':
        chosen_move = select_best_move(game_board, ai_mark='O', human_mark='X')
        player_label = "AI (O)"
    else:
        # Strategic opponent preferring center then corners
        priority_order = [4, 0, 2, 6, 8, 1, 3, 5, 7]
        chosen_move = next(m for m in priority_order if m in legal_moves)
        player_label = "Opponent (X)"

    assert chosen_move in legal_moves, f"Illegal move attempted by {player_label}"
    game_board[chosen_move] = current_turn
    print(f"Turn {turn_count:02d}: {player_label} plays at Cell {chosen_move}")
    print(format_grid(game_board))
    print()

    outcome = detect_winner(game_board)
    if outcome:
        print("-" * 65)
        print(f"Game Terminal! Result: {outcome}")
        break

    current_turn = 'O' if current_turn == 'X' else 'X'
    turn_count += 1

print("Verification: All moves were strictly legal and terminal state was accurately handled.")


SCENARIO 1: IMMEDIATE WINNING MOVE SELECTION
Current Board:
   0   1   2
0  O | O |  
  ---+---+---
1  X | X |  
  ---+---+---
2    |   |  
Minimax Selected Move: Cell 2 (Expected: 2 to win)
   0   1   2
0  O | O | O
  ---+---+---
1  X | X |  
  ---+---+---
2    |   |  
Verification: Immediate winning move selected and validated successfully.

SCENARIO 2: IMMEDIATE BLOCKING MOVE SELECTION
Current Board:
   0   1   2
0  O |   |  
  ---+---+---
1  X | X |  
  ---+---+---
2    |   |  
Minimax Selected Move: Cell 5 (Expected: 5 to block opponent)
   0   1   2
0  O |   |  
  ---+---+---
1  X | X | O
  ---+---+---
2    |   |  
Verification: Threat blocked successfully.

SCENARIO 3: FULL SIMULATED GAMEPLAY (AI 'O' VS DETERMINISTIC 'X')
Initial Board:
   0   1   2
0    |   |  
  ---+---+---
1    |   |  
  ---+---+---
2    |   |  
-----------------------------------------------------------------
Turn 01: Opponent (X) plays at Cell 4
   0   1   2
0    |   |  
  ---+---+---
1    | X |  
  ---+---